In [ ]:
import hpgeom as hpg
import numpy as np
import skyproj
import ultraplot as uplt

In [ ]:
import os
import tempfile

from rubin_sim import maf


def make_year_band_maps(opsdb, nside):
    run_name = os.path.split(opsdb)[-1].replace('.db', '')

    metric = maf.CountMetric('observationStartMJD', metric_name='NVisits')
    slicer = maf.HealpixSlicer(nside=nside)
    bundles = {}
    for band in ('u', 'g', 'r', 'i', 'z', 'y', 'all'):
        for year in np.arange(1, 11, 1):
            constraint = (
                f"scheduler_note not like '%DD%' and "
                f"observation_reason not like '%ddf%' and "
                f"observation_reason not like '%too%' and "
                f"visitExposureTime >= 20 and night <= {year} * 365 "
            )
            info_label = f"year {year}"
            if band != "all":
                constraint = f"{constraint} and band = '{band}'"
                info_label = f"{info_label} band {band}"
            bundles[f"{year} {band}"] = maf.MetricBundle(metric, slicer, constraint, run_name=run_name, info_label=info_label)

    with tempfile.TemporaryDirectory() as tmpdir:
        g = maf.MetricBundleGroup(bundles, opsdb, out_dir=str(tmpdir))
        g.run_all()

        maps = {}
        for key, value in bundles.items():
            maps[key] = value.metric_values.filled(fill_value=np.nan)

    return maps

db_path = "/Users/beckermr/RUBIN_SIM_DATA_DIR/matts_extra_sims/double_shift_v5.3.7_10yrs.db"
nside = 32
maps = make_year_band_maps(db_path, nside)

In [ ]:
lon, lat, xymap = skyproj.hpx_utils.hpxmap_to_xy(
    maps["1 all"],
    [0, 360],
    [-90, 90],
    xsize=200,
)

fig, axs = uplt.subplots(proj="moll", figsize=(6, 3))
m = axs.pcolormesh(
    lon, lat, xymap,
    cmap="viridis",
    discrete=False,
)
axs.invert_xaxis()
fig.colorbar(m, label="# of visits", labelrotation=90)

## Plot a Histograms of DESI Years

In [ ]:
from desi_stripes import get_desi_stripes, is_in_ra_range

max_visit = 100
desi_data = get_desi_stripes()
fac = 2

y3_median_visits_53_per_band = [10, 16, 40, 41, 33, 34]

fig, axs = uplt.subplots(nrows=5, ncols=6, figsize=(6 * fac, 5 * fac))


for rowind, desi_year in enumerate([1, 2, 3, 4, 5]):
    desi_inds_to_plot = np.where(desi_data["yrs"] == desi_year)[0]
    rubin_year = desi_year + 1

    for colind, band in enumerate(["u", "g", "r", "i", "z", "y"]):

        pixel_sets = []
        for desi_ind in desi_inds_to_plot:
            pixels = hpg.query_box(
                nside,
                desi_data["ramins"][desi_ind],
                desi_data["ramaxs"][desi_ind],
                desi_data["decmins"][desi_ind],
                desi_data["decmaxs"][desi_ind],
                inclusive=True,
                nest=False,
                lonlat=True,
                degrees=True
            )
            pixel_sets.append({int(pix) for pix in pixels})

        mname = f"{rubin_year:d} {band}"
        for desi_ind, pixels_to_keep in zip(desi_inds_to_plot, pixel_sets):
            data = maps[mname][np.fromiter(pixels_to_keep, dtype=np.int_)]
            axs[rowind, colind].hist(
                data,
                bins=np.linspace(-0.5, max_visit + 0.5, max_visit + 2),
                histtype="step",
                label=desi_data["names"][desi_ind],
                lw=1.5,
            )
        axs[rowind, colind].legend(loc="ur", frameon=False, ncols=1)

        if band == "u":
            axs[rowind, colind].axvline(30, color="k", linestyle="dashed")
        else:
            axs[rowind, colind].axvline(y3_median_visits_53_per_band[colind], color="k", linestyle="dashed")

        if rowind == 0:
            axs[rowind, colind].format(title=band)
        if rowind == 4:
            axs[rowind, colind].format(xlabel="# of visits")

        if colind == 0:
            axs[rowind, colind].format(ylabel=f"Rubin Y{rubin_year:d} / DESI Y{desi_year:d}")